# 01. Очистка данных об электрогитарах

**Вход:** `data/raw/` (данные с сайтов продавцов электрогитар в РФ)
**Выход:** `data/processed/guitars_clean.csv`

В ноутбуке разбираем проблемы данных по одной. Рабочий код лежит
в `src/transform.py`, здесь мы его вызываем и проверяем результат.

## План
1. Загрузка
2. Названия столбцов
3. Объединение дублирующихся столбцов
4. Фильтрация строк
5. Удаление неинформативных столбцов
6. Приведение значений в столбцах к общему виду

In [89]:
import pandas as pd
import sqlite3
import json
import sys

sys.path.append("..")

from src.transform import (normalize_columns_names, empty_to_nan, unite_duplicate_columns,
                           rename_columns, drop_sparse_columns, drop_service_columns,
                            drop_invalid_price, drop_guitars_with_expensive_items, parse_price,
                            drop_zero_price, drop_guitar_duplicates, drop_invalid_types,
                            rename_brands, parse_fret_count, rename_countries,
                            rename_neck_joints, parse_weight, drop_extra,
                            rename_anchor, parse_scale_length, rename_body_shape,
                            drop_guitars_with_wrong_body_shape, rename_orientation, rename_string_number,
                            rename_profile, rename_finish, rename_hardware, 
                            rename_fretboard, rename_series, rename_bridge,
                            rename_tuners, rename_neck_material, rename_pickups,
                            rename_controls, rename_electronics, rename_top,
                            extract_brand_series_from_title, filter_electric_guitars)

conn = sqlite3.connect("../data/raw/data.db")
def get_df(shop_name: str):   
    with conn:
        cur = conn.execute(f"SELECT json_data from raw_data WHERE shop_name = ?", (shop_name, ))
        json_list = [json.loads(row[0]) for row in cur.fetchall()]

    return pd.DataFrame(json_list)

## 1. Загрузка
Читаем сырые данные с сайтов и объединяем в одну таблицу.

In [90]:
with conn:
    cur = conn.execute("SELECT DISTINCT shop_name FROM raw_data")

dfs = [get_df(i[0]) for i in cur.fetchall()]
df = pd.concat(dfs)
print(df.shape)
df.head(5)


(10761, 589)


,Название,Цена,Код товара:,Тип:,Производитель:,Серия:,Состояние:,Страна производства:,Форма корпуса:,Ориентация:,...,Ширина грифа в 22 ладу,Регуляторы,В комплекте,струны,Звукосниматели (нековый/бриджевый),Звукосниматели (нековый/средний/бриджевый),Тип комплекта,Серия электрогитары,Отделка корпуса,Материал порожка
0,Электрогитара J&D Guitars LC100 Les Paul Gold,38 990 ₽,557973,Электрогитара,J&D Guitars,LP Series,Новое,Китай,Les Paul,под правую руку,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Электрогитара Cort KX307MS Open Pore Mahogany,38 430 ₽,506748,Электрогитара,Cort,KX,Новое,Индонезия,Double cut,под правую руку,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Электрогитара Cort KX307MS Open Pore Black,38 430 ₽,506749,Электрогитара,Cort,KX,Новое,NaN,Double cut,под правую руку,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Электрогитара Ibanez RG Gio GRG170DX Black Night,37 690 ₽,483720,Электрогитара,Ibanez,RG Gio,Новое,Китай,Superstrat,под правую руку,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Электрогитара Spira Guitars S-450 TPP Trans Pu...,37 550 ₽,595915,Электрогитара,Spira Guitars,450 Series,Новое,Китай,Double cut,под правую руку,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Названия столбцов

Сейчас в таблице находятся столбцы, названия которых не приведены к единому виду (`СЕРИЯ`, `Серия:`, `Серия` и т.д.)

Возпользуемся функцией `normalize_columns_names`. Она вернёт новый `DataFrame`, колонки которых будут в нижнем регистре и не будут содержать лишних символов.

In [91]:
print("До нормализации:", df.columns.nunique())
df = normalize_columns_names(df)
print("После нормализации:", df.columns.nunique())

До нормализации: 589
После нормализации: 476


После нормализации уникальных названий стало меньше,
однако часть одинаковых признаков всё ещё названа по-разному
(`производитель` и `бренд`). 

## 3. Объединение дублирующихся столбцов

На этом этапе нужно решить две задачи:

1. Привести названия столбцов, которые содержат одинаковый смысл, к единому названию.
2. Объединить столбцы с одинаковыми названиями.




Для корректного объединения заменим пустые и пробельный строки на np.nan.

In [92]:
df = empty_to_nan(df)

Приведём столбцы с одинаковой информацией к единому названию.

In [93]:
df = rename_columns(df)

Объединим стобцы с одинаковыми названиями.

In [94]:
df = unite_duplicate_columns(df)

Было стобцов: 589.
Стало столбцов: 241.


## 4. Фильтрация строк

Сначала удалим те строки, в которых содержится некорректная цена.

In [95]:
df = drop_invalid_price(df)

Удалено строк: 47.


Преобразуем цены в числа.

In [96]:
df = parse_price(df)

Удалим строки с нулевой ценой.

In [97]:
df = drop_zero_price(df)

Удалено строк: 4.


Удалим гитары с неподходящим типом (`бас`, `полуакустическая`, `детская`, `смарт` и т.д.). 

In [98]:
df = drop_invalid_types(df)

Удалено строк: 84


Удалим гитары, в комплектации которых есть дорогие вещи такие как `комбоусилитель` или `наушники`.
Функция `drop_guitars_with_expensive_items`.

In [99]:
df = drop_guitars_with_expensive_items(df)

Удалено строк: 97.


Уберем дубликаты.

In [100]:
df = drop_guitar_duplicates(df)

Удалено дубликатов: 1290.


Наконец, избавимся от столбцов, которые не характеризуют сами электрогитары.

## 5. Удаление неинформативных столбцов

Нужно удалить `служебные` столбцы (`артикул` и т.д.) и столбы, которые плохо заполнены (< 5%).

Удали разреженные столбцы.

In [101]:
df = drop_sparse_columns(df)

Удалено столбцов: 199.


Теперь удалим служебные столбцы.

In [102]:
df = drop_service_columns(df)


Удалено 3 столбцов.


Наконец, избавимся от столбцов, которые не характеризуют сами электрогитары.

In [103]:
df = drop_extra(df)

Удалено столбцов: 5.


Удали столбцы `цвет`, `инкрустация_грифа`, `ширина_порожка`, `материал_порожка` , ведь они никак не влияет на цену электрогитары.

In [104]:
df = df.drop(columns=["цвет", "инкрустация_грифа", "материал_порожка", "ширина_порожка", "радиус_накладки_общий",
"ширина_грифа", "переключатель"])

## 6. Приведение значений в столбцах к единому виду

Заменим символ переноса строки `\n` на `;` во всех значениях `df`.

In [105]:
df = df.replace("\n", ";", regex=True)

Извлечём производителя и серию электрогитар из названия.

In [106]:
df = extract_brand_series_from_title(df)

Оставим в датасете только электрогитары (не бас, не полуакустика). Также отсеим детские, мини и электрогитры со встроенными усилителями.

In [107]:
df = filter_electric_guitars(df)

Приведём названия производителей к единому виду функцией `rename_brands`.

In [108]:
df = rename_brands(df)

Было уникальных значений: 131.
Стало уникальных значений: 123.


Приведём значения в столбце `количество_ладов` к числовому виду.

In [109]:
df = parse_fret_count(df)

Было уникальных значений: 36.
Стало уникальных значений: 6.


Приведём значения в столбце `страна_производства` к общему виду.

In [110]:
df = rename_countries(df)

Было уникальных значений: 15.
Стало уникальных значений: 11.


Приведём значения из столбца `крепление_грифа` к единому виду.

In [111]:
df = rename_neck_joints(df)

Было уникальных значений: 28.
Стало уникальных значений: 3.


Приведём значения из столбца `вес` к числовому виду.

In [112]:
df = parse_weight(df)

Было уникальных значений: 126.
Стало уникальных значений: 125


Приведём значения из столбца `анкер` к единому виду.

In [113]:
df = rename_anchor(df)

Было уникальных значений: 12.
Стало уникальных значений: 3.


Переведём в дюймы значения из столбцов `мензура_общая` и `мензура_дюймы` и объединим их в столбец `мензура_дюймы`.

In [114]:
df = parse_scale_length(df)

Было уникальных значений: 118
Стало уникальных значений: 32.


Приведём значения из столбца `анкер` к единому виду.

In [115]:
df = rename_body_shape(df)

Было уникальных значений: 94
Стало уникальных значений: 18.


Удалим все гитары, которые имеют тип корпуса, не характерный электрогитарам.

In [116]:
df = drop_guitars_with_wrong_body_shape(df)

Удалено строк: 6


Приведём значения из столбца `ориентация` к единому виду.

In [117]:
df = rename_orientation(df)

Было уникальных значений: 9.
Стало уникальных значений: 2.


Приведём значения из столбца `количество_струн` к единому виду.

In [118]:
df = rename_string_number(df)

Было уникальных значений: 7.
Стало уникальных значений: 5.


Приведём значения из столбца `профиль_грифа` к единому виду.

In [119]:
df = rename_profile(df)

Было уникальных значений: 66.
Стало уникальных значений: 10.


Приведём значения из столбца `отделка_корпуса` к единому виду.

In [120]:
df = rename_finish(df)

Было уникальных значений: 24.
Стало уникальных значений: 3.


Приведём значения из столбца `фурнитура` к единому виду.

In [121]:
df = rename_hardware(df)

Было уникальных значений: 17.
Стало уникальных значений: 4.


Приведём значения из столбца `материал_накладки` к единому виду.

In [122]:
df = rename_fretboard(df)

Было уникальных значений: 115.


Стало уникальных значений: 15.


Приведём значения из столбца `серия` к единому виду.

In [123]:
df = rename_series(df)

Было уникальных значений: 680.
Стало уникальных значений: 566.


Приведём значения из столбца `бридж` к единому виду.

In [124]:
df = rename_bridge(df)

Было уникальных значений: 118.
Стало уникальных значений: 11.


Приведём значения из столбца `колки` к единому виду.

In [125]:
df = rename_tuners(df)

Было уникальных значений: 64.
Стало уникальных значений: 11.


Приведём значения из столбца `материал_грифа` к единому виду.

In [126]:
df = rename_neck_material(df)

Было уникальных значений: 129.
Стало уникальных значений: 13.


Приведём значения из столбца `электроника` к единому виду.

In [127]:
df = rename_electronics(df)

Было уникальных значений: 24.
Стало уникальных значений: 2.


Соединим столбцы `звукосниматели` и `конфигурация звукоснимателей` и приведём значения к единому виду. Модель звукоснимателей учитываться не будет. Важна лишь конфигурация.

In [128]:
df = rename_pickups(df)

Было уникальных значений: 95.
Стало уникальных значений: 16.


Приведём значения из столбца `управление` к единому виду. 

In [129]:

df = rename_controls(df)

Было уникальных значений: 106.
Стало уникальных значений: 22.


Приведём значения из столбца `материал_топа` к единому виду.

In [130]:
df = rename_top(df)

Было уникальных значений: 64.
стало уникалбьных значений: 17.


In [131]:
df.shape

(6701, 25)

## Заключение

После очистки данных осталось `6701` строк и `25 столбцов`. Созраним очищенный датасет в файл `../data/processed/data_clean.csv`.

In [132]:
from pathlib import Path

out_path = Path("../data/processed/guitars_clean.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False, encoding="utf-8")